<a href="https://colab.research.google.com/github/Oaimtac/farm-soccer/blob/main/%E8%82%8C%E9%9B%BB%E8%A8%8A%E8%99%9F%E9%87%8F%E6%B8%AC%E5%AF%A6%E4%BD%9C%E5%96%AE%E5%85%83_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **肌電訊號量測實作單元_3：教練這一組我真的不行了！不信你看我的肌電訊號！**

> 在這次的實驗中，將會帶大家分析兩組肌電訊號，一組是運動初期尚未產生肌肉疲勞的肌電訊號，另一組則是運動後期累積肌肉疲勞後的肌電訊號。

> 我們會採用頻域能量的角度觀察肌電訊號的頻域資訊。並嘗試用中位頻率分析法(MF)和平均功率分析法(MPF)來分析這兩組肌電訊號，進而用客觀的數據來觀察肌肉疲勞的現象。

# 0. 先收錄一些會用到的公式和函式吧！

In [ ]:
#@title
from scipy.fft import fft, fftfreq, ifft #從scipy.fft函式庫中，引入頻域轉換公式fft, fftfreq, ifft
import plotly.graph_objects as go     #引入plotly.graph_objects函式庫，命名為go
import numpy as np             #引入numpy函式庫，命名為np
import pandas as pd            #引入pandas函式庫，命名為np
from google.colab import files      #從google colab函式庫中，引入files函式

#強大的濾波器，不需要再轉換到頻域即可快速濾除雜訊，還能達到一樣的效果！
from scipy.signal import butter, filtfilt #從scipy.signal函式庫中，引入濾波器函式butter, filtfilt
#訊號處理常用的濾波器，可直接根據定義的頻域能量範圍，將正確的訊號過濾出來
#使用方法需要提供(時域訊號、資料取樣頻率、特定保留頻率起點、特定保留頻率終點)
def super_filter(data, frequency, save_frequency_start, save_frequency_end):   #super_filter(時域訊號, 週期, 特定保留頻率起點, 特定保留頻率終點)
  b, a = butter(3, [save_frequency_start, save_frequency_end], fs=frequency, btype='band')
  y = filtfilt(b, a, data)
  return y

# 1.先將自己持續出力的肌電訊號丟進程式庫吧！

In [ ]:
#先將資料夾中的「範例肌電訊號_第一組運動」和「範例肌電訊號_第三組運動」檔案放到Google雲端處理器中
uploaded = files.upload()            #用uploaded來處理上傳檔案程序
for fn in uploaded.keys():           #將選取所有上傳檔案的名字印出
  print('你已經上傳了','"{name}" '.format(
      name=fn, length=len(uploaded[fn])))

# 2. 分別畫出原始與濾波後的肌電訊號長什麼樣子吧！

In [ ]:
df = pd.read_csv('範例肌電訊號_第一組運動.txt') #以pandas的read_csv即可讀取檔案
data = np.array(df)              #將讀取到的檔案換成我們習慣的numpy陣列來處理
data2 = data[:,0]         #指定data中的第一行資料，建立陣列data2
#稍微設定一下畫布資訊！
fig = go.Figure()      #建立一個圖形物件
fig.add_trace(go.Scatter(   #新增一條線條在此圖形
    y=data2,        #新增一條線條，將取得的data2畫出
))
fig.update_layout(                  #更新圖形的說明
    title="範例肌電訊號_第一組運動 (原始訊號)",        #幫這張圖形物件命名
)
fig.show()                      #顯示圖形

period = 1/1000     #宣告資料取樣週期參數
frequency = 1000     #宣告資料取樣頻率參數
data3 = super_filter(data2, frequency, 20, 400)

fig2 = go.Figure()      #建立一個圖形物件
fig2.add_trace(go.Scatter(   #新增一條線條在此圖形
    y=data3,        #新增一條線條，將取得的data3畫出
))
fig2.update_layout(                  #更新圖形的說明
    title="範例肌電訊號_第一組運動 (濾波後訊號)",        #幫這張圖形物件命名
)
fig2.show()                     #顯示圖形

# 3. 將濾波完的肌電訊號透過頻域轉換公式，取得頻域能量分布圖吧！

> 複習頻域轉換公式，並試著取得肌電訊號的頻域能量分布圖吧！

> 取得頻域資訊後，試著計算出該段資料的中位頻率MF(median frequency)和平均功率頻率MPF(middle power frequency)吧！

In [ ]:
dots = len(data3)                #取得資料總數
period = 1/1000                  #宣告週期參數

xf = fftfreq(dots, period)[1: int(dots/2)]   #fftfreq為頻域x軸的轉換公式，需要加入y值訊號所含的點數與週期，轉換後得出頻譜分布圖的x值陣列。
                          #頻域轉換公式只需取前二分之一有效值

yf = fft(data3)                  #fft 為頻域y軸的轉換公式，轉換後得出頻譜分布圖的y值陣列，
yf_half = np.abs(yf[1: int(dots/2)])        #頻域轉換公式只需取前二分之一有效值
yf_normalized = 2 / dots * yf_half        #頻域轉換後將資料正規畫

fig = go.Figure()                 #建立一個圖形物件
fig.add_trace(go.Scatter(x=xf, y=yf_normalized))  #新增一條線條在此圖形，將xf, yf_normalized在x, y軸上畫出
fig.show()                     #顯示圖形

In [ ]:
#根據公式，取得MF最重要的第一件事就是要知道所有頻率能量加總是多少！
#之後只要再依序將頻率能量累加，搭配二分之一頻率能量加總，就能取得中位數頻率能量的位置囉！

fullSum = np.sum(yf_normalized) #將所有能量加總
halfSum = 0  #宣告一個二分之一能量總和的累加變數
counter = 0     #累加到二分之一能量總和時，紀錄該頻率抵達MF位置

for i in yf_normalized:     #迴圈依序計算頻率能量
  if(halfSum < fullSum/2):   #halfSum累加至fullSum的二分之一，會抵達MF位置(一半能量總和)
    halfSum = halfSum + i  #依序累加頻率能量至halfSum
    counter = counter + 1  #紀錄累加的次數
  else:
    break            #halfSum累加至fullSum的二分之一後，脫離迴圈，
MF = xf[counter]         #根據紀錄累加的次數推算在xf上的位置，即為MF的頻率
print('整段資料的Median Frequency為', MF,'赫茲',)

In [ ]:
#根據公式，計算MPF即是把所有頻率能量對應頻率做加權平均，再除以總能量！

fullSum = np.sum(yf_normalized) #將所有能量加總
counter = 0           #累加到該頻率時，紀錄該頻率位置
frequency_power = 0       #宣告一個頻率乘以頻率能量的總和累加變數
for i in yf_normalized:                 #迴圈依序計算頻率能量
  frequency_power = frequency_power + i*xf[counter] #依序累加頻率乘以頻率能量至frequency_power
  counter = counter + 1               #紀錄累加的次數

MPF = frequency_power / fullSum

print('整段資料的Middle Power Frequency為', MPF,'赫茲',)

In [ ]:
#自行練習
#根據第一段運動的肌電訊號，紀錄下MF前期與MPF前期
#再根據第三段運動的肌電訊號，紀錄下MF後期與MPF後期

MF前期 =
MPF前期 =
MF後期 =
MPF後期 =

#-----------參考疲勞指數的計算方法，推算出疲勞指數(MF)與疲勞指數(MPF)----------#






#------------------------------------------------------------------------------#

# 小結：
---

1.   肌電訊號的頻域分析雖在學理上有明確的指引，但實際要讓肌肉感到疲乏，並在肌電訊號上反應，並非透過單次的施力與計算就能達到顯著的分析效果。

2.   利用較高強度的運動，並分析運動前期與後期的訊號，較能取得明顯的肌肉疲乏證據。

3.   MF和MPF雖有著完全不同的公式，但其物理意義皆是反應中低頻的能量變化，因此兩種指標的升降趨勢雷同。